In [134]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [135]:
df_links=pd.read_csv(r'C:\Users\Shubham Joshi\Desktop\CineFlicx\artifacts\ingested\ml-latest-small\links.csv')
df_movies=pd.read_csv(r'C:\Users\Shubham Joshi\Desktop\CineFlicx\artifacts\ingested\ml-latest-small\movies.csv')
df_ratings=pd.read_csv(r'C:\Users\Shubham Joshi\Desktop\CineFlicx\artifacts\ingested\ml-latest-small\ratings.csv')
df_tags=pd.read_csv(r'C:\Users\Shubham Joshi\Desktop\CineFlicx\artifacts\ingested\ml-latest-small\tags.csv')

In [136]:
df_links.shape

(9742, 3)

In [137]:
df_movies.shape

(9742, 3)

In [138]:
df_ratings.shape


(100836, 4)

In [139]:
df_tags.shape

(3683, 4)

In [140]:
col_link=df_links.columns
col_link

Index(['movieId', 'imdbId', 'tmdbId'], dtype='str')

In [141]:
col_movie=df_movies.columns
col_movie

Index(['movieId', 'title', 'genres'], dtype='str')

In [142]:
col_rating=df_ratings.columns
col_rating

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')

In [143]:
col_tag=df_tags.columns
col_tag

Index(['userId', 'movieId', 'tag', 'timestamp'], dtype='str')

In [144]:
df_tags.rename(columns={'tag_id':'tag'},inplace=True)

In [145]:
df_tags.rename(
    columns={
        'userId': 'userid',
        'movieId': 'movieid'
    },
    inplace=True
)

In [146]:
df_movies.rename(
    columns={
        
        'movieId': 'movieid'
    },
    inplace=True
)

In [147]:
df_links.rename(
    columns={
        'tmdbId': 'tmdbid',
        'movieId': 'movieid',
        'imdbId': 'imdbid',
    },
    inplace=True
)

In [148]:
df_ratings.rename(
    columns={
        'movieId': 'movieid',
        'userId': 'userid',
        
    },
    inplace=True
)

In [149]:
print(df_links.isnull().sum())
print(df_links.duplicated().sum())

movieid    0
imdbid     0
tmdbid     8
dtype: int64
0


In [150]:
print(df_tags.isnull().sum())
print(df_tags.duplicated().sum())

userid       0
movieid      0
tag          0
timestamp    0
dtype: int64
0


In [151]:
print(df_movies.isnull().sum())
print(df_movies.duplicated().sum())

movieid    0
title      0
genres     0
dtype: int64
0


In [152]:
print(df_ratings.isnull().sum())
print(df_ratings.duplicated().sum())

userid       0
movieid      0
rating       0
timestamp    0
dtype: int64
0


In [153]:
df_links.dropna(subset=['tmdbid'], inplace=True)

In [154]:
df_tags_grouped = (
    df_tags
    .groupby("movieid")["tag"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

In [155]:
df_movies_links = pd.merge(
    df_movies,
    df_links,
    on="movieid",
    how="left"
)

In [156]:
df_metadata = pd.merge(
    df_movies_links,
    df_tags_grouped,
    on="movieid",
    how="left"
)

In [157]:
df_metadata.columns

Index(['movieid', 'title', 'genres', 'imdbid', 'tmdbid', 'tag'], dtype='str')

In [158]:
df_metadata["combined_features"] = (
    df_metadata["title"].fillna('') + " " +
    df_metadata["genres"].fillna('') + " " +
    df_metadata["tag"].fillna('')
)

In [159]:
print(df_metadata.shape)
print(df_ratings.shape)

(9742, 7)
(100836, 4)


In [160]:
df_ratings.columns

Index(['userid', 'movieid', 'rating', 'timestamp'], dtype='str')

In [161]:
threshold = 50

active_users = df_ratings['userid'].value_counts()
active_users = active_users[active_users > threshold].index

df_ratings = df_ratings[df_ratings['userid'].isin(active_users)]

In [162]:
print(df_ratings['userid'].nunique())
print(df_ratings.shape)

378
(93462, 4)


In [164]:
movieid_to_title = dict(
    zip(df_metadata["movieid"], df_metadata["title"])
)

In [165]:
title_to_movieid = dict(
    zip(df_metadata["title"], df_metadata["movieid"])
)

In [166]:
movie_counts = df_ratings.groupby("movieid")["rating"].count()

popular_movies = movie_counts[movie_counts >= 50].index

df_ratings = df_ratings[
    df_ratings["movieid"].isin(popular_movies)
]
df_ratings.shape

(34925, 4)

In [167]:
movie_pivot = df_ratings.pivot_table(
    index="movieid",
    columns="userid",
    values="rating"
).fillna(0)

In [ ]:
movie_similarity = cosine_similarity(movie_pivot)

In [169]:
similarity_df = pd.DataFrame(
    movie_similarity,
    index=movie_pivot.index,
    columns=movie_pivot.index
)

In [186]:
similarity_df.columns

Index([     1,      2,      6,     10,     11,     16,     17,     19,     21,
           25,
       ...
        78499,  79132,  80463,  81845,  89745,  91500,  91529,  99114, 109487,
       112852],
      dtype='int64', name='movieid', length=408)

In [187]:
def recommend(movie_title, top_n=5):

    movie_id = title_to_movieid[movie_title]

    if movie_id not in similarity_df.index:
        return "Movie not available in filtered dataset"

    similar_movies = similarity_df.loc[movie_id].sort_values(
        ascending=False
    )[1:top_n+1]

    recommended_movie_ids = similar_movies.index

    recommendations = df_metadata[
        df_metadata["movieid"].isin(recommended_movie_ids)
    ]

    return recommendations

In [189]:
recommend('Shawshank Redemption, The (1994)')

,movieid,title,genres,imdbid,tmdbid,tag,combined_features
257,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,110912.0,680.0,good dialogue great soundtrack non-linear cult...,Pulp Fiction (1994) Comedy|Crime|Drama|Thrille...
314,356,Forrest Gump (1994),Comedy|Drama|Romance|War,109830.0,13.0,shrimp Vietnam bubba gump shrimp lieutenant da...,Forrest Gump (1994) Comedy|Drama|Romance|War s...
461,527,Schindler's List (1993),Drama|War,108052.0,424.0,moving thought-provoking Holocaust based on a ...,Schindler's List (1993) Drama|War moving thoug...
510,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,102926.0,274.0,Hannibal Lector disturbing drama gothic psycho...,"Silence of the Lambs, The (1991) Crime|Horror|..."
2226,2959,Fight Club (1999),Action|Crime|Drama|Thriller,137523.0,550.0,dark comedy psychology thought-provoking twist...,Fight Club (1999) Action|Crime|Drama|Thriller ...
